# Навчання детекторів UXO — Google Colab (GPU)
## Покращена конфігурація: 20 епох, imgsz=640, focal loss, cos_lr

### Що покращено порівняно з локальним CPU-навчанням

| Параметр | Базова (CPU, 15 ep) | Colab (GPU, 20 ep) | Ефект |
|---|---|---|---|
| `imgsz` | 416 | **640** | Більше деталей → краще знаходить дрібні UXO |
| `batch` | 2 | **16** | Стабільніший градієнт, швидше навчання |
| `flipud` | 0.0 | **0.5** | Вертикальний фліп: супутник не має «верху» |
| `degrees` | 0.0 | **15** | Ротація ±15°: аерознімки під будь-яким кутом |
| `copy_paste` | 0.0 | **0.1** | Копіювання UXO-об'єктів між зображеннями |
| `fl_gamma` | 0.0 | **1.5** | Focal loss: більше уваги рідкісним UXO-прикладам |
| `cos_lr` | False | **True** | Косинусний LR: краща збіжність |
| `lr0` | 0.01 | **0.001** | Стабільніше навчання |
| `conf` (оцінка) | 0.25 | **0.10** | Менше пропусків UXO (↓FNR) |

### Датасет
Використовується **`warehouse_focused`** (87 МБ zip): 3050 зображень, лише `uxo` + `destruction`.
Він точніший за повний warehouse: 5.1 анотацій/зображення проти 2.1, без vehicle-шуму.

Файл **`warehouse_focused.zip`** лежить у папці `project data folder/` — завантажте його на Google Drive:
`MyDrive/uxo-detection-data/warehouse_focused.zip`

Notebook також підтримує `warehouse.zip` (повний датасет) — якщо покласти поруч, він підхопиться автоматично.

### Порядок запуску
1. `Runtime → Change runtime type → T4 GPU`
2. Завантажте `warehouse_focused.zip` на Drive (`MyDrive/uxo-detection-data/`)
3. Запускайте комірки по черзі (Shift+Enter)

In [ ]:
# 1. Встановлення залежностей та перевірка GPU
!pip install ultralytics -q

import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print('УВАГА: GPU недоступний! Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# 2. Підключення Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive підключено')

In [ ]:
# 3. Розпакування даних з Drive
# Шукає warehouse_focused.zip (рекомендовано) або warehouse.zip
import zipfile, shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/uxo-detection-data')
COLAB_RUNS = Path('/content/runs_improved')
COLAB_RUNS.mkdir(parents=True, exist_ok=True)

def _extract_or_copy(zip_name, drive_root):
    """Розпаковує zip або копіює папку. Повертає Path або None."""
    local_dir = Path('/content') / zip_name.replace('.zip', '')
    if local_dir.exists():
        return local_dir
    zip_src = drive_root / zip_name
    dir_src = drive_root / zip_name.replace('.zip', '')
    if zip_src.exists():
        print(f'Розпаковую {zip_src.name} ...')
        with zipfile.ZipFile(str(zip_src), 'r') as z:
            z.extractall('/content/')
        if local_dir.exists():
            return local_dir
    elif dir_src.exists():
        print(f'Копіюю {dir_src.name} ...')
        shutil.copytree(str(dir_src), str(local_dir))
        return local_dir
    return None

# Пробуємо focused спочатку, потім повний warehouse
COLAB_WAREHOUSE = (
    _extract_or_copy('warehouse_focused.zip', DRIVE_ROOT) or
    _extract_or_copy('warehouse.zip', DRIVE_ROOT)
)

if COLAB_WAREHOUSE is None:
    raise FileNotFoundError(
        f'Не знайдено warehouse_focused.zip або warehouse.zip у {DRIVE_ROOT}\n'
        'Завантажте warehouse_focused.zip (87 МБ) на Google Drive у папку uxo-detection-data.'
    )

IS_FOCUSED = 'focused' in COLAB_WAREHOUSE.name
print(f'\nДатасет: {COLAB_WAREHOUSE.name}  |  {"nc=2 (uxo+destruction)" if IS_FOCUSED else "nc=3 (uxo+destruction+vehicle)"}')
print('Статистика:')
for split in ['train', 'val', 'test']:
    n_img = len(list((COLAB_WAREHOUSE / 'images' / split).glob('*.*')))
    n_lbl = len(list((COLAB_WAREHOUSE / 'labels' / split).glob('*.txt')))
    print(f'  {split:5s}: {n_img:5d} зображень, {n_lbl:5d} міток')

In [ ]:
# 4. Створення data.yaml (nc визначається автоматично)
import yaml

DATA_YAML = Path('/content/data.yaml')

# warehouse_focused: тільки uxo + destruction (nc=2)
# warehouse повний:  uxo + destruction + vehicle (nc=3)
if IS_FOCUSED:
    nc, names = 2, ['uxo', 'destruction']
else:
    nc, names = 3, ['uxo', 'destruction', 'vehicle']

cfg = {
    'path':  str(COLAB_WAREHOUSE),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    nc,
    'names': names,
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print('data.yaml:')
print(open(DATA_YAML).read())

In [ ]:
# 5. Конфігурація навчання (покращені гіперпараметри)
import torch, time

DEVICE    = '0' if torch.cuda.is_available() else 'cpu'
EPOCHS    = 20
IMGSZ     = 640
BATCH     = 16 if DEVICE == '0' else 4
CONF_EVAL = 0.10   # нижче 0.25 → менше пропусків UXO

print(f'Device: {DEVICE} | Epochs: {EPOCHS} | ImgSz: {IMGSZ} | Batch: {BATCH} | conf_eval: {CONF_EVAL}')

# Перевіряємо версію ultralytics
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')

TRAIN_KWARGS = dict(
    data     = str(DATA_YAML),
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    device   = DEVICE,
    project  = str(COLAB_RUNS),
    patience = 20,
    plots    = True,
    verbose  = True,
    save     = True,
    # --- Augmentations для супутникових/аерознімків ---
    flipud   = 0.5,    # вертикальний фліп (у супутника немає 'верху')
    fliplr   = 0.5,    # горизонтальний фліп
    degrees  = 15.0,   # ротація ±15°
    mosaic   = 1.0,
    mixup    = 0.05,
    hsv_s    = 0.7,
    hsv_v    = 0.4,
    scale    = 0.5,
    # --- LR schedule ---
    lr0           = 0.001,
    lrf           = 0.01,
    cos_lr        = True,   # косинусний LR
    warmup_epochs = 3,
    # --- Loss / регуляризація ---
    cls          = 1.5,      # збільшуємо вагу classification loss (замість fl_gamma)
    weight_decay = 0.0005,
    amp          = True,
    close_mosaic = 5,
)

print('Гіперпараметри налаштовано.')

## Навчання YOLOv8n (nano, ~3.2M параметрів)

In [ ]:
# 6. YOLOv8n
from ultralytics import YOLO

t0 = time.time()
model_n = YOLO('yolov8n.pt')
res_n   = model_n.train(name='yolov8n_improved', **TRAIN_KWARGS)
print(f'YOLOv8n: {(time.time()-t0)/60:.1f} хв  |  ваги: {res_n.save_dir}/weights/best.pt')

## Навчання YOLOv8s (small, ~11.2M параметрів)

In [ ]:
# 7. YOLOv8s
t0 = time.time()
model_s = YOLO('yolov8s.pt')
res_s   = model_s.train(name='yolov8s_improved', **TRAIN_KWARGS)
print(f'YOLOv8s: {(time.time()-t0)/60:.1f} хв  |  ваги: {res_s.save_dir}/weights/best.pt')

## Навчання RT-DETR-L (transformer, ~20M параметрів)
> **Примітка:** RT-DETR потребує більше GPU-пам'яті. Якщо виникає OOM — зменшіть `batch` до 4 або пропустіть цю комірку.

In [ ]:
# 8. RT-DETR
from ultralytics import RTDETR

rt_kwargs = {**TRAIN_KWARGS}
rt_kwargs['batch'] = 8 if DEVICE == '0' else 2   # RT-DETR важчий за пам'яттю

t0 = time.time()
try:
    model_rt = RTDETR('rtdetr-l.pt')
    res_rt   = model_rt.train(name='rtdetr_improved', **rt_kwargs)
    print(f'RT-DETR: {(time.time()-t0)/60:.1f} хв  |  ваги: {res_rt.save_dir}/weights/best.pt')
    rtdetr_ok = True
except Exception as e:
    print(f'RT-DETR не вдалось: {e}')
    res_rt   = None
    rtdetr_ok = False

## Оцінка моделей на тестовій вибірці
Використовуємо `conf=0.10` замість 0.25 — це знижує FNR (частку пропущених UXO).

In [ ]:
# 9. Оцінка
import pandas as pd
import numpy as np

trained_models = {
    'YOLOv8n': Path(res_n.save_dir) / 'weights' / 'best.pt',
    'YOLOv8s': Path(res_s.save_dir) / 'weights' / 'best.pt',
}
if rtdetr_ok and res_rt is not None:
    trained_models['RT-DETR'] = Path(res_rt.save_dir) / 'weights' / 'best.pt'

eval_rows = {}
for mname, wpath in trained_models.items():
    if not wpath.exists():
        print(f'  {mname}: best.pt не знайдено, пропускаємо')
        continue
    print(f'Оцінюємо {mname} (conf={CONF_EVAL}) ...')
    m   = YOLO(str(wpath))
    val = m.val(data=str(DATA_YAML), split='test', conf=CONF_EVAL,
                iou=0.5, device=DEVICE, plots=False)
    p = float(val.box.mp)
    r = float(val.box.mr)
    eval_rows[mname] = {
        'mAP@0.5':      float(val.box.map50),
        'mAP@0.5:0.95': float(val.box.map),
        'Precision':    p,
        'Recall':       r,
        'F2':           (5*p*r) / (4*p + r + 1e-9),
        'FNR':          1.0 - r,
    }

df_new = pd.DataFrame(eval_rows).T
print('\n=== Результати покращених моделей (conf=0.10) ===')
print(df_new.round(3).to_string())

## Порівняння: базові (CPU) vs покращені (GPU) результати

In [ ]:
# 10. Порівняння з базовими результатами (зі збережених метрик)
import matplotlib.pyplot as plt

# Базові результати (CPU, 15 ep, imgsz=416, conf=0.25, warehouse повний nc=3)
baseline = {
    'YOLOv8n': {'mAP@0.5': 0.083, 'Recall': 0.094, 'F2': 0.106},
    'YOLOv8s': {'mAP@0.5': 0.137, 'Recall': 0.178, 'F2': 0.192},
    'RT-DETR': {'mAP@0.5': 0.022, 'Recall': 0.036, 'F2': 0.044},
}
df_base = pd.DataFrame(baseline).T

# Тільки моделі, які є в обох
common = [m for m in df_base.index if m in df_new.index]
if not common:
    print('Немає спільних моделей для порівняння')
else:
    df_b = df_base.loc[common]
    df_n = df_new.loc[common]

    metrics = ['mAP@0.5', 'Recall', 'F2']
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    x = np.arange(len(common))
    w = 0.35

    for ax, metric in zip(axes, metrics):
        b_vals = df_b[metric].values
        n_vals = df_n[metric].values
        bars_b = ax.bar(x - w/2, b_vals, w, label='Базова (CPU, 15ep, conf=0.25)',
                        color='#FF7043', alpha=0.85)
        bars_n = ax.bar(x + w/2, n_vals, w, label='Покращена (GPU, 20ep, conf=0.10)',
                        color='#4CAF50', alpha=0.85)
        for bar in list(bars_b) + list(bars_n):
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(common, fontsize=11)
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    note = ' (nc=2, без vehicle)' if IS_FOCUSED else ' (nc=3)'
    plt.suptitle(f'Базова (CPU, nc=3) vs Покращена (GPU{note})', fontsize=12)
    plt.tight_layout()
    plt.show()

    print('\nПриріст показників:')
    for name in common:
        for metric in metrics:
            old = df_b.loc[name, metric]
            new = df_n.loc[name, metric]
            delta = new - old
            sign = '+' if delta >= 0 else ''
            print(f'  {name:8s} {metric:12s}: {old:.3f} -> {new:.3f}  ({sign}{delta:.3f})')

In [ ]:
# 11. Збереження результатів на Google Drive
import shutil, zipfile

DRIVE_RESULTS  = DRIVE_ROOT / 'runs_colab_improved'
DRIVE_MODELS   = DRIVE_ROOT / 'models_trained'   # сюди кладемо best.pt
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

# Копіюємо всі runs
shutil.copytree(str(COLAB_RUNS), str(DRIVE_RESULTS), dirs_exist_ok=True)

# Зберігаємо CSV
df_new.to_csv(str(DRIVE_RESULTS / 'comparison_improved.csv'))

# Копіюємо best.pt кожної моделі у DRIVE_MODELS у структурі, сумісній з main.ipynb:
#   notebooks/models/trained/{name}_improved/weights/best.pt
name_map = {
    'yolov8n_improved': 'yolov8n_uxo_improved',
    'yolov8s_improved': 'yolov8s_uxo_improved',
    'rtdetr_improved':  'rtdetr_uxo_improved',
}
saved_pts = []
for run_dir in sorted(COLAB_RUNS.iterdir()):
    pt = run_dir / 'weights' / 'best.pt'
    if not pt.exists():
        continue
    out_name = name_map.get(run_dir.name, run_dir.name)
    dest = DRIVE_MODELS / out_name / 'weights' / 'best.pt'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(pt), str(dest))
    saved_pts.append(dest)
    print(f'  {dest.relative_to(DRIVE_ROOT)}  ({pt.stat().st_size/1e6:.1f} MB)')

# Зіп усіх ваг для зручного завантаження
zip_out = DRIVE_RESULTS / 'weights_improved.zip'
with zipfile.ZipFile(str(zip_out), 'w', zipfile.ZIP_DEFLATED) as zf:
    for pt in saved_pts:
        zf.write(str(pt), arcname=str(pt.relative_to(DRIVE_ROOT)))
print(f'\nzip для завантаження: {zip_out}')
print(f'Розпакуйте weights_improved.zip у папку notebooks/models/trained/ проекту')
print('\nГотово. Результати збережено:', DRIVE_RESULTS)